# Triage_HF Statistical & Graphical Analytical Model
* Started date: 02/07/2024 - 05:09 AM
* Data Management Team, Triage_HF

# 1. Dataset cleaning
We will import our dataset and perform a first cleanup to begin to understand which columns and values we are dealing with.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import datetime as dt
import plotly.io as pio
#pio.renderers.default = "browser"
pd.options.mode.chained_assignment = None
pd.set_option('display.max.rows', None)
pd.set_option('display.max.columns', None)

There are two ways to import our data set:
1. Locally: if we have the repository locally, we will use the following line

In [2]:
# Locally method
train_df = pd.read_csv('../dataset/raw/TRIAGE_2024.csv')

2. Remotely: if we want to connect the dataset from github, we must import it using the following line



**WARNING**: every time we want to use this form, we must generate the token again.

In [3]:
# Github method
# train_df = pd.read_csv('https://raw.githubusercontent.com/Adriellevy/Triage_HF/main/Data%20Analysis%20and%20Reports/dataset/raw/TRIAGE%202024.csv?token=GHSAT0AAAAAACLM5VZGUTNJJPTGADJMMUKEZOQ3JLA')

In [4]:
train_df.head()

,3+-99999|a,[ñ_MJ,NOMBRE Y APELLIDO,MOTIVO DE CONSULTA,BOX,TRIAGE,ENFERMERO,MEDICO,DESTINO,A,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,FECHA: 01/01/2024 ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,01-01,LUSI,FIEBRE Y TOS,18,IV,ERIKA,RODRIGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,01-01,LO PINTO CARLOS,FIEBRE Y TOS,17,IV,SOLEDAD G,SOLEDAD,alta,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,01-01,BANDERA,TOS,12,IV,ERIKA,RODRIGO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,01-01,TOMINO EDUARDO,HTA,12,IV,SOLEDAD G,SOLEDAD,ALTA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1.1 Columns rename

At first, you can see how certain columns are wrongly named, so I will assign a corresponding name to them:

In [5]:
cols_rename = { '[ñ_MJ': 'FECHA DE INGRESO',
                'A': 'AISLADO' }
train_df.rename(columns = cols_rename, inplace = True)

In addition, each time you move to the next day according to the date of entry, the first column returns to 1:

In [6]:
train_df.loc[train_df['FECHA DE INGRESO'] == '01-01', '3+-99999|a'].iloc[0] == train_df.loc[train_df['FECHA DE INGRESO'] == '01-02', '3+-99999|a'].iloc[0]

True

Therefore, I will name that first column "NUMERO DE TURNO" in reference to the fact that the turns are reset at the beginning of the next day:

In [7]:
col_rename = { '3+-99999|a': 'NUMERO DE TURNO' }
train_df.rename(columns = col_rename, inplace = True)

## 1.2 Removing headers and null rows

We will remove the header that appears every time a new day begins:

In [8]:
train_df = train_df[train_df['NUMERO DE TURNO'].str.contains('FECHA|N°') == False]

In addition, we will remove the last rows of the dataset that do not contain any value in first and last name

In [9]:
train_df = train_df[train_df['NOMBRE Y APELLIDO'].notnull()]

## 1.3 Changing Triage Level data type

For possible machine learning models in the future, the triage level should be int dtype:

In [10]:
train_df['TRIAGE'].dtype

dtype('O')

In [11]:
train_df['TRIAGE'] = train_df['TRIAGE'].str.strip().str.upper()

In [12]:
vals_rename = { 'I': 1, 'II': 2, 'III': 3, 'IV': 4 }
train_df['TRIAGE'] = train_df['TRIAGE'].replace(vals_rename)

Those rows not containing 1, 2, 3, or 4 will be taken as null (using errors coerce)

In [13]:
train_df['TRIAGE'] = pd.to_numeric(train_df['TRIAGE'], downcast = "signed", errors = 'coerce')

In [14]:
train_df['TRIAGE'].dtype

dtype('float64')

## 1.4 Changing Entry Date data type

We will change the object type of the entry date to date type. This makes date manipulation much easier and more readable:

In [15]:
train_df['FECHA DE INGRESO'].dtype

dtype('O')

In [16]:
train_df['FECHA DE INGRESO'] = train_df['FECHA DE INGRESO'] + '-2024'

In [17]:
train_df['FECHA DE INGRESO'] = pd.to_datetime(train_df['FECHA DE INGRESO'], format = '%d-%m-%Y', errors = 'coerce')

In [18]:
train_df['FECHA DE INGRESO'].dtype

dtype('<M8[ns]')

## 1.5 Changing Isolated data type

In [19]:
train_df['AISLADO'].unique()

array([nan, 'KPC', 'NO', 'SI', 'No', '724', 'Si', 'ECOLI METALO'],
      dtype=object)

In [20]:
data = {'AISLADO': [np.nan, 'KPC', 'NO', 'SI', 'No', '724', 'Si', 'A', 'ECOLI METALO']}
mapping = {np.nan: False, 'NO': False, 'No': False,
           'KPC': True, 'ECOLI METALO': True, 'SI': True, 'Si': True, 'A': True}
train_df['AISLADO'] = train_df['AISLADO'].map(mapping)

In [21]:
train_df = train_df[train_df['AISLADO'] != '724']

In [22]:
train_df['AISLADO'] = train_df['AISLADO'].astype(bool)

In [23]:
train_df['DESTINO'].unique()

array([nan, 'alta', 'ALTA', '624', 'ALTA ', 'FUGA', '506', '?', 'INT',
       '715', '721', '512', '717', '820', '622', 'ALT.VOL', '312', '812',
       'HMD/ 315', 'HMD', '316', '301', '806', 'Alta', '802', '809',
       'PISO', '315', '722', '805', 'alta ', 'OBITO', 'Traslado', ' ',
       '    ', '311', '808', 'ALTA VOLUNT', '302', '813', '818', '304',
       'ALTA V ', '814', '613', '619', '602', '615', 'ALTA MEDICA', '314',
       '626', '607', '726', 'ALTA VOL', '926', '922', '920', '917', '916',
       '711', 'DERIVACION', '714', '719', '  ', '724', 'FUGA ', 'DERIV',
       'AL. VOL', '915', '918', 'QUIROFANO', 'ALTA VOLUNT.', '310',
       'TRASLADO ', '928', '919', '616', 'QUIROFANO ', '628', '909',
       '804', 'ALTA VOLUNTARIA', 'DERIVACION ', '720', '803', 'DERIVAC',
       'alTA', '908', '709', '621', '306',
       'ALTA                                                                                                                                                           

## 1.6 Generating ALTA column

In [24]:
train_df = train_df[~train_df['DESTINO'].isin(['?', 'INT', 'HMD/ 315', 'HMD', ' ', '    ', '  ', ])]

In [25]:
train_df['ALTA'] = train_df['DESTINO'].str.contains(
    'alta|obito|traslado|derivacion|AL. VOL|DERIVAC',
    case=False,  
    regex=True   
)
train_df['ALTA'].fillna(True, inplace=True)

Finally, empty and unnamed columns can be visualized. We will remove them:

In [26]:
cols_to_keep = ['NUMERO DE TURNO', 'FECHA DE INGRESO', 'NOMBRE Y APELLIDO', 'MOTIVO DE CONSULTA', 'BOX', 'TRIAGE', 'ENFERMERO', 'MEDICO', 'DESTINO', 'ALTA', 'AISLADO']
train_df = train_df[cols_to_keep]

This is how our dataframe would look at first:

In [27]:
train_df.head()

,NUMERO DE TURNO,FECHA DE INGRESO,NOMBRE Y APELLIDO,MOTIVO DE CONSULTA,BOX,TRIAGE,ENFERMERO,MEDICO,DESTINO,ALTA,AISLADO
1,1,2024-01-01,LUSI,FIEBRE Y TOS,18,4.0,ERIKA,RODRIGO,NaN,True,False
2,2,2024-01-01,LO PINTO CARLOS,FIEBRE Y TOS,17,4.0,SOLEDAD G,SOLEDAD,alta,True,False
3,3,2024-01-01,BANDERA,TOS,12,4.0,ERIKA,RODRIGO,NaN,True,False
4,4,2024-01-01,TOMINO EDUARDO,HTA,12,4.0,SOLEDAD G,SOLEDAD,ALTA,True,False
5,5,2024-01-01,D IORIO ROLANDO EMILIO,FIEBRE,5,4.0,ERIKA,SOLEDAD,NaN,True,False


In [85]:
def grafico_barras(df, p_x, p_y, title, x_title, y_title, media = None):
    fig = px.bar(df, x=p_x, y=p_y, barmode="group")
    fig.update_layout(title=title, xaxis_title=x_title, yaxis_title=y_title, title_x=0.5)
    fig.update_traces(texttemplate='%{y}', textposition='outside')
    if(media):
        media = df[p_y].mean()
        fig.add_trace(go.Scatter(x=df[p_x],
                                 y=[media] * len(df),
                                 mode='lines',
                                 name='Media',
                                 line=dict(color='red', width=2, dash='dash'),
                                 hovertemplate='%{y:.2f}'))
        fig.add_annotation(
        xref='paper', yref='y',
        x=-0.03, y=media,
        text=f'{media:.2f}',
        showarrow=False,
        font=dict(color='red'))
    fig.show()


In [79]:
def filtros(df, desde = None, hasta = None, nombre_y_apellido = None, motivo_de_consulta = None, box = None, triage = None, medico = None, enfermero = None, alta = None, aislado = None):
    if(desde != None and hasta != None):
        desde = pd.to_datetime(desde, format='%d-%m-%Y', errors='coerce') 
        hasta = pd.to_datetime(hasta, format='%d-%m-%Y', errors='coerce')   
        df = df[(df['FECHA DE INGRESO'] >= desde) & (df['FECHA DE INGRESO'] <= hasta)]
        
    df['FECHA DE INGRESO'] = df['FECHA DE INGRESO'].dt.strftime('%d-%m-%Y')      
       
    if(nombre_y_apellido != None):
        df = df[df['NOMBRE Y APELLIDO'] == nombre_y_apellido]

    if(motivo_de_consulta != None):
        df = df[df['MOTIVO DE CONSULTA'] == motivo_de_consulta]

    if(box != None):
        df = df[df['BOX'] == box]

    if(triage != None):
        df = df[df['TRIAGE'] == triage]

    if(medico != None):
        df = df[df['MEDICO'] == medico]

    if(enfermero != None):
        df = df[df['ENFERMERO'] == enfermero]

    if(alta != None):
        df = df[df['ALTA'] == alta]

    if(aislado != None):
        df = df[df['AISLADO'] == aislado]

    return df

In [80]:
def cant_pacientes_fecha(desde = None, hasta = None, nombre_y_apellido = None, motivo_de_consulta = None, box = None, triage = None, medico = None, enfermero = None, alta = None, aislado = None):
    
    train_df_acotado = train_df.groupby('FECHA DE INGRESO', sort=False).size().reset_index()
    train_df_acotado.rename(columns={0: 'CANTIDAD DE PACIENTES'}, inplace=True)
    train_df_acotado = filtros(train_df_acotado, desde, hasta, nombre_y_apellido, motivo_de_consulta, box, triage, medico, enfermero, alta, aislado)
    return train_df_acotado


In [82]:
df = cant_pacientes_fecha(desde = None, hasta = None,
             nombre_y_apellido = None, motivo_de_consulta = None,
             box = None, triage = None,
             medico = None, enfermero = None,
             alta = None, aislado = None)

grafico_barras(df = df, 
               p_x='FECHA DE INGRESO', p_y='CANTIDAD DE PACIENTES', 
               x_title='Cantidad de Pacientes por Fecha de Ingreso', y_title='Fecha de Ingreso', title='Cantidad de Pacientes', 
               media=True)


In [107]:
def top_consultas_fecha(desde=None, hasta=None, top = 10, order=None, nombre_y_apellido = None, motivo_de_consulta = None, box = None, triage = None, medico = None, enfermero = None, alta = None, aislado = None):
    top_motivos = train_df.copy(deep=True)

    top_motivos = filtros(top_motivos, desde, hasta, nombre_y_apellido, motivo_de_consulta, box, triage, medico, enfermero, alta, aislado)
    
    motivos_count = top_motivos['MOTIVO DE CONSULTA'].value_counts()

    top_motivos = motivos_count.nlargest(top).reset_index()
    top_motivos.columns = ['MOTIVO DE CONSULTA', 'CANTIDAD DE CONSULTAS']

    if order == 'asc':
        top_motivos.sort_values(by='CANTIDAD DE CONSULTAS', ascending = True, inplace = True)
        
    return top_motivos

In [108]:
df = top_consultas_fecha(desde = None, hasta=None,
             top = 15, order = 'asc',
             nombre_y_apellido = None, motivo_de_consulta = None,
             box = None, triage = None,
             medico = None, enfermero = None,
             alta = None, aislado = None)

grafico_barras(df = df, 
               p_x='MOTIVO DE CONSULTA', p_y='CANTIDAD DE CONSULTAS', 
               title='Cantidad de Consultas', x_title='Motivos de Consulta mas Frecuentes', y_title='Motivo de Consulta', 
               media=False)